In [ ]:
import pdfplumber
import pdfminer
import pypdfium2

In [3]:
!pip install xhtml2pdf markdown

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 399.6 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 399.6 kB/s eta 0:00:04
   ---------------- ----------------------- 0.8/2.0 MB 435.8 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 435.8 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 435.8 kB/s eta 0:00:03
   --------------------- ------------------ 1.0/2.0 MB 430


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import glob
import re
from datetime import datetime
import markdown
from xhtml2pdf import pisa # <--- New import

# ... [Keep your existing parse_date_from_text and HTML generation code] ...

# Build a comprehensive HTML structure
# (Ensure utf-8 meta tag is included for special characters)
combined_html = """
<html>
<head>
    <meta charset="utf-8">
    <style>
        body { font-family: Helvetica, Arial, sans-serif; line-height: 1.6; margin: 40px; }
        h1 { color: #333333; text-align: center; }
        h2 { color: #2c3e50; border-bottom: 1px solid #eeeeee; padding-bottom: 5px; margin-top: 40px; }
        .meta-info { color: #7f8c8d; font-size: 0.9em; margin-bottom: 20px; font-style: italic; }
        code { background-color: #f4f4f4; padding: 2px 5px; }
        pre { background-color: #f4f4f4; padding: 10px; }
        blockquote { margin: 0 0 0 15px; color: #666666; }
    </style>
</head>
<body>
    <h1>Compiled Tech Summaries</h1>
"""

for data in file_data:
    html_content = markdown.markdown(data['content'], extensions=['tables', 'fenced_code'])
    formatted_date = data['date'].strftime('%B %d, %Y')
    extracted_context = data['date_label']
    
    combined_html += f"<h2>Report: {formatted_date}</h2>\n"
    combined_html += f"<div class='meta-info'>Extracted date context: \"{extracted_context}\" | Source: {os.path.basename(data['filepath'])}</div>\n"
    combined_html += html_content + "\n<hr>\n"

combined_html += """
</body>
</html>
"""

# Convert the HTML string into a PDF using xhtml2pdf
print(f"Generating PDF: {output_pdf}...")

try:
    # Open utility function to write the PDF
    with open(output_pdf, "w+b") as result_file:
        # convert HTML to PDF
        pisa_status = pisa.CreatePDF(
            combined_html,                # the HTML to convert
            dest=result_file,             # file handle to recieve result
            encoding='utf-8'              # force utf-8 for proper symbol rendering
        )

    # Return True on success and False on errors
    if pisa_status.err:
        print(f"Error generating PDF: {pisa_status.err}")
    else:
        print("PDF generation complete!")
        
except Exception as e:
    print(f"Exception encountered: {e}")

In [5]:
!pip install pdfkit


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Install required libraries if you haven't already:
# !pip install markdown pdfkit

"""
Note on PDF libraries: 
pdfplumber, pdfminer, and pypdfium2 are used for READING and EXTRACTING text from existing PDFs.
To CREATE a PDF from Markdown, the most robust way is converting Markdown to HTML, 
then converting HTML to PDF using pdfkit or weasyprint.
"""


md_dir = 'md files'
output_pdf = 'compiled_tasks.pdf'
md_files = glob.glob(os.path.join(md_dir, '*.md'))

def parse_date_from_text(text, filename):
    """
    Extracts dates like 'April 1–2, 2026' or 'March 14, 2026' from the text content.
    Returns a datetime object for sorting, and the extracted string for labeling.
    """
    # Regex to find Month DD, YYYY (handling ranges like DD-DD or DD–DD)
    months = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
    # Matches: "April 1-2, 2026", "March 14, 2026", "April 1â€“2, 2026"
    pattern = re.compile(rf"({months})\s+(\d{{1,2}})(?:\s*(?:[-–—]|to|â€“)\s*\d{{1,2}})?,\s*(\d{{4}})", re.IGNORECASE)
    
    match = pattern.search(text)
    if match:
        month_str, day_str, year_str = match.groups()
        try:
            # Construct a parsable date string from the first day in the range
            clean_date_str = f"{month_str} {day_str}, {year_str}"
            return datetime.strptime(clean_date_str, "%B %d, %Y"), match.group(0)
        except ValueError:
            pass
            
    # Fallback to filename if no date found in the text
    try:
        date_str = os.path.basename(filename).replace('Grok_Task_', '').replace('.md', '')
        return datetime.strptime(date_str, '%Y-%m-%dT%H-%M-%S'), "Date from filename"
    except ValueError:
        return datetime.min, "Unknown Date"

file_data = []
for file in md_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    date_obj, extracted_str = parse_date_from_text(content, file)
    file_data.append({
        'date': date_obj,
        'date_label': extracted_str,
        'filepath': file,
        'content': content
    })

# Sort files chronologically based on the extracted text dates
file_data.sort(key=lambda x: x['date'])

# Build a comprehensive HTML structure
combined_html = """
<html>
<head>
    <style>
        body { font-family: Arial, sans-serif; line-height: 1.6; margin: 40px; }
        h1 { color: #333; text-align: center; }
        h2 { color: #2c3e50; border-bottom: 2px solid #eee; padding-bottom: 5px; margin-top: 40px; }
        .meta-info { color: #7f8c8d; font-size: 0.9em; margin-bottom: 20px; font-style: italic; }
        code { background-color: #f4f4f4; padding: 2px 5px; border-radius: 3px; }
        pre { background-color: #f4f4f4; padding: 10px; border-radius: 5px; overflow-x: auto; }
        blockquote { border-left: 4px solid #ccc; margin: 0; padding-left: 15px; color: #666; }
    </style>
</head>
<body>
    <h1>Compiled Tech Summaries</h1>
"""

for data in file_data:
    # Convert markdown to HTML directly
    html_content = markdown.markdown(data['content'], extensions=['tables', 'fenced_code'])
    
    formatted_date = data['date'].strftime('%B %d, %Y')
    extracted_context = data['date_label']
    
    combined_html += f"<h2>Report: {formatted_date}</h2>\n"
    combined_html += f"<div class='meta-info'>Extracted date context: \"{extracted_context}\" | Source: {os.path.basename(data['filepath'])}</div>\n"
    combined_html += html_content + "\n<hr>\n"

combined_html += """
</body>
</html>
"""

# Convert the HTML string into a PDF
try:
    print(f"Generating PDF: {output_pdf}...")
    pdfkit.from_string(combined_html, output_pdf)
    print("PDF generation complete!")
except Exception as e:
    print("Error generating PDF:", e)
    print("\nNote: pdfkit requires wkhtmltopdf to be installed on your system.")
    print("Download it from: https://wkhtmltopdf.org/downloads.html and add it to your PATH.")

NameError: name 'file_data' is not defined

In [1]:
import os
files  = os.listdir("md files")

In [2]:
import os
import glob
import re
from datetime import datetime
import markdown
from xhtml2pdf import pisa

# Configuration
md_dir = 'md files'
output_pdf = 'compiled_tasks.pdf'
md_files = glob.glob(os.path.join(md_dir, '*.md'))

def parse_date_from_text(text, filename):
    """
    Extracts dates like 'April 1–2, 2026' or 'March 14, 2026' from the text content.
    Returns a datetime object for sorting, and the extracted string for labeling.
    """
    # Regex to find Month DD, YYYY (handling ranges like DD-DD or DD–DD)
    months = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
    pattern = re.compile(rf"({months})\s+(\d{{1,2}})(?:\s*(?:[-–—]|to|â€“)\s*\d{{1,2}})?,\s*(\d{{4}})", re.IGNORECASE)
    
    match = pattern.search(text)
    if match:
        month_str, day_str, year_str = match.groups()
        try:
            # Construct a parsable date string from the first day in the range
            clean_date_str = f"{month_str} {day_str}, {year_str}"
            return datetime.strptime(clean_date_str, "%B %d, %Y"), match.group(0)
        except ValueError:
            pass
            
    # Fallback to filename if no date found in the text
    try:
        date_str = os.path.basename(filename).replace('Grok_Task_', '').replace('.md', '')
        return datetime.strptime(date_str, '%Y-%m-%dT%H-%M-%S'), "Date from filename"
    except ValueError:
        return datetime.min, "Unknown Date"

file_data = []
for file in md_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    date_obj, extracted_str = parse_date_from_text(content, file)
    file_data.append({
        'date': date_obj,
        'date_label': extracted_str,
        'filepath': file,
        'content': content
    })

# Sort files chronologically based on the extracted text dates
file_data.sort(key=lambda x: x['date'])

# Build a comprehensive HTML structure
combined_html = """
<html>
<head>
    <meta charset="utf-8">
    <style>
        body { font-family: Helvetica, Arial, sans-serif; line-height: 1.6; margin: 40px; }
        h1 { color: #333333; text-align: center; }
        h2 { color: #2c3e50; border-bottom: 1px solid #eeeeee; padding-bottom: 5px; margin-top: 40px; }
        .meta-info { color: #7f8c8d; font-size: 0.9em; margin-bottom: 20px; font-style: italic; }
        code { background-color: #f4f4f4; padding: 2px 5px; border-radius: 3px;}
        pre { background-color: #f4f4f4; padding: 10px; border-radius: 5px;}
        blockquote { margin: 0 0 0 15px; color: #666666; border-left: 4px solid #ccc; padding-left: 15px;}
    </style>
</head>
<body>
    <h1>Compiled Tech Summaries</h1>
"""

for data in file_data:
    # Convert markdown to HTML directly
    html_content = markdown.markdown(data['content'], extensions=['tables', 'fenced_code'])
    
    formatted_date = data['date'].strftime('%B %d, %Y')
    extracted_context = data['date_label']
    
    combined_html += f"<h2>Report: {formatted_date}</h2>\n"
    combined_html += f"<div class='meta-info'>Extracted date context: \"{extracted_context}\" | Source: {os.path.basename(data['filepath'])}</div>\n"
    combined_html += html_content + "\n<hr>\n"

combined_html += """
</body>
</html>
"""

# Convert the HTML string into a PDF using xhtml2pdf
print(f"Generating PDF: {output_pdf}...")

try:
    with open(output_pdf, "w+b") as result_file:
        pisa_status = pisa.CreatePDF(
            combined_html,
            dest=result_file,
            encoding='utf-8'
        )

    if pisa_status.err:
        print(f"Error generating PDF: {pisa_status.err}")
    else:
        print("PDF generation complete!")
        
except Exception as e:
    print(f"Exception encountered: {e}")

Generating PDF: compiled_tasks.pdf...
PDF generation complete!


In [3]:
import os
import glob
import re
from datetime import datetime
import markdown
from xhtml2pdf import pisa # Or keep using pdfkit if you prefer

md_dir = 'md files'
output_pdf = 'compiled_tasks.pdf'
md_files = glob.glob(os.path.join(md_dir, '*.md'))

def parse_date_from_text(text, filename):
    # [Keep your existing parse_date_from_text function here]
    months = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
    pattern = re.compile(rf"({months})\s+(\d{{1,2}})(?:\s*(?:[-–—]|to|â€“)\s*\d{{1,2}})?,\s*(\d{{4}})", re.IGNORECASE)
    
    match = pattern.search(text)
    if match:
        month_str, day_str, year_str = match.groups()
        try:
            clean_date_str = f"{month_str} {day_str}, {year_str}"
            return datetime.strptime(clean_date_str, "%B %d, %Y"), match.group(0)
        except ValueError:
            pass
            
    try:
        date_str = os.path.basename(filename).replace('Grok_Task_', '').replace('.md', '')
        return datetime.strptime(date_str, '%Y-%m-%dT%H-%M-%S'), "Date from filename"
    except ValueError:
        return datetime.min, "Unknown Date"

file_data = []
for file in md_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    date_obj, extracted_str = parse_date_from_text(content, file)
    file_data.append({
        'date': date_obj,
        'date_label': extracted_str,
        'filepath': file,
        'content': content
    })

file_data.sort(key=lambda x: x['date'])

# --- UPGRADED CSS & HTML STRUCTURE ---
combined_html = """
<html>
<head>
    <meta charset="utf-8">
    <style>
        @page {
            margin: 2cm;
        }
        body { 
            font-family: Georgia, 'Times New Roman', serif; 
            line-height: 1.7; 
            color: #2b2b2b; 
            background-color: #fcfbf8; /* Subtle off-white/parchment */
            margin: 0;
            padding: 0;
        }
        h1 { 
            color: #1a1a1a; 
            text-align: center; 
            text-transform: uppercase; 
            letter-spacing: 2px; 
            border-bottom: 2px solid #3e2723; /* Deep academic brown */
            padding-bottom: 10px;
            font-weight: bold;
            margin-bottom: 50px;
        }
        h2 { 
            color: #3e2723; 
            border-bottom: 1px solid #d4c5b9; 
            padding-bottom: 8px; 
            margin-top: 40px; 
            font-size: 1.5em;
        }
        h3 {
            color: #4a3b32;
            font-style: italic;
            font-size: 1.2em;
        }
        .meta-info { 
            color: #5c534d; 
            font-size: 0.9em; 
            margin-bottom: 25px; 
            font-style: italic; 
            text-align: left;
            padding: 5px 10px;
            background-color: #f0eadd;
            border-left: 3px solid #8b7355;
        }
        a {
            color: #63432b;
            text-decoration: none;
            font-weight: bold;
        }
        code { 
            font-family: 'Courier New', Courier, monospace;
            background-color: #2b2b2b; 
            color: #dcdcaa; /* VS Code-style gold/green for inline code */
            padding: 2px 6px; 
            border-radius: 2px; 
            font-size: 0.9em;
        }
        pre { 
            background-color: #1e1e1e; /* Dark mode code block */
            color: #e6e1d8; 
            padding: 15px; 
            border-radius: 4px; 
            border-left: 4px solid #8b7355;
        }
        pre code {
            background-color: transparent;
            color: inherit;
            padding: 0;
        }
        blockquote { 
            border-left: 4px solid #8b7355; 
            margin: 20px 0; 
            padding: 10px 20px; 
            background-color: #f2efe9; 
            color: #3d3530; 
            font-style: italic; 
        }
        ul, ol {
            padding-left: 25px;
        }
        li {
            margin-bottom: 8px;
        }
        hr.divider {
            border: 0;
            border-top: 1px dashed #d4c5b9;
            margin: 40px 0;
        }
    </style>
</head>
<body>
    <h1>Compiled Tech Summaries</h1>
"""

for data in file_data:
    html_content = markdown.markdown(data['content'], extensions=['tables', 'fenced_code'])
    
    formatted_date = data['date'].strftime('%B %d, %Y')
    extracted_context = data['date_label']
    
    combined_html += f"<h2>Report: {formatted_date}</h2>\n"
    combined_html += f"<div class='meta-info'><strong>Context:</strong> {extracted_context} | <strong>Source:</strong> {os.path.basename(data['filepath'])}</div>\n"
    combined_html += html_content + "\n<hr class='divider'>\n"

combined_html += """
</body>
</html>
"""

print(f"Generating PDF: {output_pdf}...")

try:
    with open(output_pdf, "w+b") as result_file:
        pisa_status = pisa.CreatePDF(
            combined_html,
            dest=result_file,
            encoding='utf-8'
        )

    if pisa_status.err:
        print(f"Error generating PDF: {pisa_status.err}")
    else:
        print("PDF generation complete!")
        
except Exception as e:
    print(f"Exception encountered: {e}")

Generating PDF: compiled_tasks.pdf...
PDF generation complete!


In [ ]:
import os
import glob
import re
from datetime import datetime
import markdown
from xhtml2pdf import pisa

md_dir = 'md files'
output_pdf = 'compiled_tasks.pdf'
md_files = glob.glob(os.path.join(md_dir, '*.md'))

def parse_date_from_text(text, filename):
    months = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
    pattern = re.compile(rf"({months})\s+(\d{{1,2}})(?:\s*(?:[-–—]|to|â€“)\s*\d{{1,2}})?,\s*(\d{{4}})", re.IGNORECASE)
    
    match = pattern.search(text)
    if match:
        month_str, day_str, year_str = match.groups()
        try:
            clean_date_str = f"{month_str} {day_str}, {year_str}"
            return datetime.strptime(clean_date_str, "%B %d, %Y"), match.group(0)
        except ValueError:
            pass
            
    try:
        date_str = os.path.basename(filename).replace('Grok_Task_', '').replace('.md', '')
        return datetime.strptime(date_str, '%Y-%m-%dT%H-%M-%S'), "Date from filename"
    except ValueError:
        return datetime.min, "Unknown Date"

file_data = []
for file in md_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    date_obj, extracted_str = parse_date_from_text(content, file)
    file_data.append({
        'date': date_obj,
        'date_label': extracted_str,
        'filepath': file,
        'content': content
    })

file_data.sort(key=lambda x: x['date'])

# --- 19TH CENTURY ACADEMIC CSS & HTML STRUCTURE ---
combined_html = """
<html>
<head>
    <meta charset="utf-8">
    <style>
        @page {
            margin: 2.5cm;
            background-color: #f0e6d2;
        }
        body { 
            background-color: #f0e6d2; /* Explicitly set on body so xhtml2pdf applies it */
            font-family: 'Book Antiqua', Palatino, 'Palatino Linotype', Georgia, serif; 
            font-size: 14pt; /* Increased base text size */
            line-height: 1.6; 
            color: #2c1e16; /* Deep sepia/charcoal ink */
            margin: 0;
            padding: 0;
        }
        h1 { 
            color: #1a110b; 
            text-align: center; 
            text-transform: uppercase; 
            letter-spacing: 3px; 
            border-bottom: 3px double #5c3a21; /* Classic double rule */
            padding-bottom: 15px;
            font-weight: normal; /* Rely on size and spacing rather than bolding */
            font-size: 26pt;
            margin-bottom: 40px;
        }
        h2 { 
            color: #3d2616; 
            border-bottom: 1px solid #8c6d53; 
            padding-bottom: 5px; 
            margin-top: 40px; 
            font-size: 18pt;
            font-style: italic; /* Victorian academic feel */
        }
        h3 {
            color: #4a3b32;
            font-size: 15pt;
            font-weight: bold;
        }
        .meta-info { 
            color: #5c534d; 
            font-size: 11pt; 
            margin-bottom: 25px; 
            font-style: italic; 
            text-align: left;
            padding: 8px 12px;
            background-color: #e6d8c3; /* Slightly darker parchment for contrast */
            border-left: 4px solid #8b7355;
        }
        a {
            color: #5c3a21;
            text-decoration: none;
            border-bottom: 1px dashed #5c3a21;
        }
        code { 
            font-family: 'Courier New', Courier, monospace;
            background-color: #e6d8c3; 
            color: #4a2e1b; 
            padding: 2px 6px; 
            border: 1px solid #d1bfae;
            font-size: 12pt;
        }
        pre { 
            background-color: #2b2622; /* Dark slate/brown for code blocks */
            color: #dcd3c6; 
            padding: 15px; 
            border-left: 5px solid #8b7355;
            font-size: 12pt;
        }
        pre code {
            background-color: transparent;
            color: inherit;
            border: none;
            padding: 0;
        }
        blockquote { 
            border-left: 4px solid #8b7355; 
            margin: 20px 0; 
            padding: 10px 20px; 
            background-color: #e6d8c3; 
            color: #4a3b32; 
            font-style: italic; 
        }
        ul, ol {
            padding-left: 30px;
        }
        li {
            margin-bottom: 10px;
        }
        hr.divider {
            border: 0;
            border-top: 2px dashed #8c6d53;
            margin: 50px 0;
        }
    </style>
</head>
<body>
    <h1>Compiled Technical Summaries</h1>
"""

for data in file_data:
    html_content = markdown.markdown(data['content'], extensions=['tables', 'fenced_code'])
    
    formatted_date = data['date'].strftime('%B %d, %Y')
    extracted_context = data['date_label']
    
    combined_html += f"<h2>Report: {formatted_date}</h2>\n"
    combined_html += f"<div class='meta-info'><strong>Extracted Context:</strong> {extracted_context} <br> <strong>Source Archive:</strong> {os.path.basename(data['filepath'])}</div>\n"
    combined_html += html_content + "\n<hr class='divider'>\n"

combined_html += """
</body>
</html>
"""

print(f"Generating PDF: {output_pdf}...")

try:
    with open(output_pdf, "w+b") as result_file:
        pisa_status = pisa.CreatePDF(
            combined_html,
            dest=result_file,
            encoding='utf-8'
        )

    if pisa_status.err:
        print(f"Error generating PDF: {pisa_status.err}")
    else:
        print("PDF generation complete!")
        
except Exception as e:
    print(f"Exception encountered: {e}")

Generating PDF: compiled_tasks.pdf...
PDF generation complete!


In [2]:
files[0:5]

['Grok_Task_2026-04-02T17-04-11.md',
 'Grok_Task_2026-04-02T17-05-00.md',
 'Grok_Task_2026-04-02T17-05-06.md',
 'Grok_Task_2026-04-02T17-05-11.md',
 'Grok_Task_2026-04-02T17-05-19.md']